# 11 · Train / Test / Evaluation Split
Define the standard dataset split for CityPulse audio modeling:
- Train: UrbanSound8K Folds 1–9 (5,753 clips)
- Test: UrbanSound8K Fold 10 (605 clips)
- Evaluation / OOD: ESC-50 (200 clips)

In [1]:
# Cell 1 – interpreter setup & imports
import sys
from pathlib import Path
import hashlib

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

import numpy as np
import pandas as pd
print('Successfully imported numpy and pandas')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported numpy and pandas


In [2]:
# Cell 2 – verify sha256 of inputs, load labels and features
def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

labels_path = REPO_ROOT / 'data/processed/labels.csv'
features_path = REPO_ROOT / 'data/processed/audio_features.npy'

labels_sha = compute_sha256(labels_path)
features_sha = compute_sha256(features_path)

print(f'labels.csv sha256         : {labels_sha}')
print(f'audio_features.npy sha256 : {features_sha}')

EXPECTED_LABELS_SHA = '8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854'
EXPECTED_FEAT_SHA   = 'ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a'

assert labels_sha == EXPECTED_LABELS_SHA, f'labels.csv SHA mismatch: {labels_sha}'
assert features_sha == EXPECTED_FEAT_SHA, f'audio_features.npy SHA mismatch: {features_sha}'

labels = pd.read_csv(labels_path)
X = np.load(features_path)

print(f'labels rows : {len(labels):,}')
print(f'X shape     : {X.shape}')

assert len(labels) == 6558, f'Expected 6558 rows in labels, got {len(labels)}'
assert X.shape == (6558, 128), f'Expected X.shape == (6558, 128), got {X.shape}'
print('Cell 2 SHA256 asserts and loads passed ✓')

labels.csv sha256         : 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
audio_features.npy sha256 : ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a
labels rows : 6,558
X shape     : (6558, 128)
Cell 2 SHA256 asserts and loads passed ✓


In [3]:
# Cell 3 – create split indices and verify sizes, disjointness, and full coverage
train_idx = labels.index[(labels['source'] == 'us8k') & (labels['fold'] <= 9)].to_numpy()
test_idx = labels.index[(labels['source'] == 'us8k') & (labels['fold'] == 10)].to_numpy()
esc_idx = labels.index[labels['source'] == 'esc50'].to_numpy()

print(f'train_idx size : {len(train_idx)} (expect 5753)')
print(f'test_idx size  : {len(test_idx)} (expect 605)')
print(f'esc_idx size   : {len(esc_idx)} (expect 200)')

assert len(train_idx) == 5753, f'Expected len(train_idx) == 5753, got {len(train_idx)}'
assert len(test_idx) == 605, f'Expected len(test_idx) == 605, got {len(test_idx)}'
assert len(esc_idx) == 200, f'Expected len(esc_idx) == 200, got {len(esc_idx)}'

s_train = set(train_idx)
s_test = set(test_idx)
s_esc = set(esc_idx)

# Disjointness checks
assert len(s_train & s_test) == 0, f'train_idx and test_idx overlap by {len(s_train & s_test)} items'
assert len(s_train & s_esc) == 0, f'train_idx and esc_idx overlap by {len(s_train & s_esc)} items'
assert len(s_test & s_esc) == 0, f'test_idx and esc_idx overlap by {len(s_test & s_esc)} items'

# Full coverage check
union_all = s_train | s_test | s_esc
assert len(union_all) == 6558, f'Expected union size 6558, got {len(union_all)}'
assert union_all == set(range(6558)), 'Union of index sets does not equal set(range(6558))'

print('Cell 3 asserts passed: disjoint sets covering all 6,558 rows ✓')

train_idx size : 5753 (expect 5753)
test_idx size  : 605 (expect 605)
esc_idx size   : 200 (expect 200)
Cell 3 asserts passed: disjoint sets covering all 6,558 rows ✓


In [4]:
# Cell 4 – assert no ESC-50 rows in train/test and verify fold sets
assert (labels.loc[train_idx, 'source'] == 'us8k').all(), 'Found non-us8k row in train_idx'
assert (labels.loc[test_idx, 'source'] == 'us8k').all(), 'Found non-us8k row in test_idx'
assert (labels.loc[esc_idx, 'source'] == 'esc50').all(), 'Found non-esc50 row in esc_idx'

train_folds = set(labels.loc[train_idx, 'fold'].unique())
test_folds = set(labels.loc[test_idx, 'fold'].unique())

print(f'Train folds : {sorted(train_folds)}')
print(f'Test folds  : {sorted(test_folds)}')

assert train_folds == set(range(1, 10)), f'Expected train folds {{1..9}}, got {train_folds}'
assert test_folds == {10}, f'Expected test folds {{10}}, got {test_folds}'
print('Cell 4 asserts passed: source and fold isolation verified ✓')

Train folds : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Test folds  : [np.int64(10)]
Cell 4 asserts passed: source and fold isolation verified ✓


In [5]:
# Cell 5 – print per app_class counts for train and test and assert exact distributions
train_counts = labels.loc[train_idx, 'app_class'].value_counts().sort_index().to_dict()
test_counts = labels.loc[test_idx, 'app_class'].value_counts().sort_index().to_dict()

print('=== Train (US8K Folds 1-9) per app_class counts ===')
for cls, cnt in sorted(train_counts.items()):
    print(f'  {cls:20s}: {cnt:4d}')

print('\n=== Test (US8K Fold 10) per app_class counts ===')
for cls, cnt in sorted(test_counts.items()):
    print(f'  {cls:20s}: {cnt:4d}')

EXPECTED_TRAIN = {
    'construction': 1804,
    'hvac': 900,
    'music_loudspeaker': 900,
    'siren': 846,
    'traffic_horn': 1303
}

EXPECTED_TEST = {
    'construction': 196,
    'hvac': 100,
    'music_loudspeaker': 100,
    'siren': 83,
    'traffic_horn': 126
}

assert train_counts == EXPECTED_TRAIN, f'Train counts mismatch: expected {EXPECTED_TRAIN}, got {train_counts}'
assert test_counts == EXPECTED_TEST, f'Test counts mismatch: expected {EXPECTED_TEST}, got {test_counts}'
print('\nCell 5 asserts passed: exact class distributions verified ✓')

=== Train (US8K Folds 1-9) per app_class counts ===


  construction        : 1804
  hvac                :  900
  music_loudspeaker   :  900
  siren               :  846
  traffic_horn        : 1303

=== Test (US8K Fold 10) per app_class counts ===
  construction        :  196
  hvac                :  100
  music_loudspeaker   :  100
  siren               :   83
  traffic_horn        :  126

Cell 5 asserts passed: exact class distributions verified ✓


In [6]:
# Cell 6 – majority-class baseline accuracy on test set
most_common_train_class = labels.loc[train_idx, 'app_class'].value_counts().idxmax()
test_classes = labels.loc[test_idx, 'app_class']

test_correct = int((test_classes == most_common_train_class).sum())
total_test = len(test_idx)
baseline_accuracy = test_correct / total_test

print(f'Most common training class : {most_common_train_class}')
print(f'Correct test predictions   : {test_correct} / {total_test}')
print(f'Majority baseline accuracy : {baseline_accuracy:.4f} ({baseline_accuracy * 100:.2f}%)')

assert most_common_train_class == 'construction', f'Expected most common class construction, got {most_common_train_class}'
assert test_correct == 196, f'Expected 196 correct, got {test_correct}'
assert total_test == 605, f'Expected 605 total test, got {total_test}'
print('Cell 6 asserts passed ✓')

Most common training class : construction
Correct test predictions   : 196 / 605
Majority baseline accuracy : 0.3240 (32.40%)
Cell 6 asserts passed ✓


In [7]:
# Cell 7 – save split_indices.npz, reload in fresh variables, and verify
out_path = REPO_ROOT / 'data/processed/split_indices.npz'
np.savez(out_path, train_idx=train_idx, test_idx=test_idx, esc_idx=esc_idx)
print(f'Saved split_indices.npz to: {out_path}')

# Reload in fresh variables
with np.load(out_path) as reloaded:
    reloaded_train = reloaded['train_idx']
    reloaded_test = reloaded['test_idx']
    reloaded_esc = reloaded['esc_idx']

    assert np.array_equal(reloaded_train, train_idx), 'Reloaded train_idx does not match in-memory array'
    assert np.array_equal(reloaded_test, test_idx), 'Reloaded test_idx does not match in-memory array'
    assert np.array_equal(reloaded_esc, esc_idx), 'Reloaded esc_idx does not match in-memory array'

print('Reloaded arrays match in-memory arrays exactly ✓')

npz_sha = compute_sha256(out_path)
print(f'split_indices.npz sha256: {npz_sha}')
print('Cell 7 asserts passed ✓')

Saved split_indices.npz to: C:\Users\LENOVO\Desktop\CityPulse\data\processed\split_indices.npz
Reloaded arrays match in-memory arrays exactly ✓
split_indices.npz sha256: 2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b
Cell 7 asserts passed ✓
